In [1]:
using LowLevelFEM, LinearAlgebra

[ Info: Precompiling LowLevelFEM [6171b9fb-adbf-4751-adb9-5faded75de07](cache misses: include_dependency fsize change (1), wrong dep version loaded (2), wrong source (1), incompatible header (6))
[ Info: Precompiling LowLevelFEM [6171b9fb-adbf-4751-adb9-5faded75de07] (cache misses: include_dependency fsize change (2), wrong dep version loaded (4), wrong source (2), incompatible header (12))

SYSTEM: caught exception of type :MethodError while trying to print a failed Task notice; giving up


In [2]:
openGeometry("boxes.geo")

In [3]:
#openPreProcessor()

In [4]:
mat = Material("body")
U = Field([mat], type=:VectorField, dim=3, fieldName=:u);

In [5]:
bc_bottom = BoundaryCondition("bottom", ux=0, uy=0, uz=0)
bc_top = BoundaryCondition("top", ux=0, uz=0, uy=(x,y,z)->-x*(x-10) * z*(z-10) / 2000)

K = ∫(SymGrad(U) ⋅ D(:Solid, mat) ⋅ SymGrad(U))
f = ∫(U ⋅ [0, 0, 0])

@time u = solveField(Symmetric(K), f, support=[bc_bottom, bc_top])

showDoFResults(u, name="u", factor=1, visible=true)

  9.836301 seconds (19.70 M allocations: 1.006 GiB, 3.02% gc time, 94.45% compilation time)


0

In [6]:
@time C = contact(u, master="master", slave="slave", leaf_size=1)

  5.860658 seconds (7.63 M allocations: 371.266 MiB, 0.87% gc time, 96.60% compilation time)


Contact("slave" -> "master", 1438 candidate nodes, 743 active, G=(4314, 12765), C=(4314, 4314))

In [7]:
@time updateContact!(C, 1.3u)

  0.918633 seconds (721.85 k allocations: 39.573 MiB, 82.27% compilation time)


Contact("slave" -> "master", 1438 candidate nodes, 875 active, G=(4314, 12765), C=(4314, 4314))

In [8]:
@time updateContact!(C, 1.01u)

  0.172589 seconds (168.43 k allocations: 13.341 MiB, 12.32% gc time)


Contact("slave" -> "master", 1438 candidate nodes, 746 active, G=(4314, 12765), C=(4314, 4314))

In [9]:
for ls in (1, 2, 4, 8)
    @time contact(
        u,
        master="master",
        slave="slave",
        leaf_size=ls
    )
end

  0.188808 seconds (166.72 k allocations: 13.107 MiB)
  0.194396 seconds (153.92 k allocations: 12.662 MiB)
  0.219731 seconds (142.62 k allocations: 12.248 MiB)
  0.288070 seconds (136.22 k allocations: 12.002 MiB, 15.02% gc time)


In [10]:
C.gap

elementwise ScalarField
[[0.12038307489217742; 0.11285909212346285; 0.10890764440182953;;], [0.11293802726421981; 0.12038307489217742; 0.10890764440182953;;], [0.11187091088202081; 0.11969717883400638; 0.10850135748963934;;], [0.11969717883400638; 0.1123399210625751; 0.10850135748963934;;], [0.11244468991678426; 0.11970741812733265; 0.10871458464337447;;], [0.11970741812733265; 0.1124465221043103; 0.10871458464337447;;], [0.12068343538586113; 0.11303484790472251; 0.1095079634495247;;], [0.1132113012436753; 0.12068343538586113; 0.1095079634495247;;], [0.10474644854110832; 0.11187091088202081; 0.10020661098290863;;], [0.10020661098290863; 0.11187091088202081; 0.10850135748963934;;]  …  [-0.11864665971116384; -0.13181308774668266; -0.12185019601696162;;], [0.0437003560761482; 0.055447806382233344; 0.05505603636721715;;], [0.047290649051913346; 0.0437003560761482; 0.05505603636721715;;], [-0.04355596382459695; -0.03994028121633206; -0.024264667532152007;;], [0.0737858352178073; 0.063848539

In [11]:
C.G[:,:]

4314×12765 SparseArrays.SparseMatrixCSC{Float64, Int64} with 28760 stored entries:
⎡⢿⣿⠲⠄⠀⢸⠀⠀⠀⣺⣷⣿⠀⠀⢠⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎤
⎢⠀⠀⠀⠀⠀⠘⣿⣿⣿⣿⣿⡏⠀⠀⠀⠙⢦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⢨⣿⣿⣿⣿⣿⣷⠀⠀⠀⠀⠀⠙⢦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠀⠀⠀⠀⠀⢸⢿⣿⣿⣿⣿⣿⠀⠀⠀⠀⠀⠀⠀⠙⢦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⢨⣿⠀⠀⠀⢸⣿⣿⣿⣿⣿⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎢⠈⡟⠀⠀⠀⢸⣿⣿⣿⣿⣿⣿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⢦⡀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎥
⎣⠺⠿⠀⠀⠀⠸⠉⠉⠻⠿⠿⠿⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠙⠦⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⠀⎦

In [12]:
showElementResults(C.n, name="n")

1

In [13]:
showElementResults(C.t1, name="t1")

2

In [14]:
showElementResults(C.t2, name="t2")

3

In [15]:
showElementResults(C.gap, name="gap", visible=false)

4

In [16]:
C.active

1438-element BitVector:
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0
 ⋮
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0
 0

In [17]:
C.state

LoadError: FieldError: type Contact has no field `state`, available fields: `master`, `slave`, `U`, `multiplier`, `displacement`, `slave_nodes`, `master_element_tags`, `master_local_coordinates`, `master_points`, `gap`, `gap_values`, `g`, `G`, `C`, `E`, `n`, `t1`, `t2`, `active`, `cn`, `ct`, `cn_values`, `ct_values`, `multiplier_dofs`, `options`

In [19]:
openPostProcessor()